In [2]:
# Biblioteca
import arcpy
import os
from arcpy.sa import * 

# Geodatabase
gdb = "F:\Projetos_ArcGIS_Gama\Qualidade_da_Agua\Qualidade_da_Agua.gdb"

# projeto atual
aprx = arcpy.mp.ArcGISProject("CURRENT")
mapa = aprx.activeMap  # mapa ativo no ArcGIS Pro

In [4]:
# Filtro e Kernel para 10 anos - OS Qualidade da Água

In [2]:
# Entradas
input_layer = "BDGIS.Ordens_Servico"
anos = range(2015, 2026)

# lista para adicionar os Rasters no Contents ao final do processo
rasters_gerados = []

for ano in anos:
    
    print(f"\n🔄 Processando ano {ano}...")
    
    # Criar filtro
    sql_filter = f"""
    DATAEXECUCAOINICIO between '{ano}-01-01 00:00:00' and '{ano}-12-31 23:59:59'
    AND DESCRICAORA in ('Aguas Claras', 'Agua Quente', 'Arniqueira', 'Brazlandia', 'Ceilandia', 'Ceilandia II',
    'Gama', 'Recanto das Emas', 'Riacho Fundo', 'Riacho Fundo II', 'Samambaia', 'Santa Maria', 'Taguatinga')
    AND MOTIVONAOEXECUCAO = 0
    AND DESCSITUACAOOS = 'Baixada'
    AND SERVAPROPRIADO in ('8400108011302')
    """
    
    filtered_layer = f"OS_Filtradas_{ano}"
    
    arcpy.MakeFeatureLayer_management(input_layer, filtered_layer, sql_filter)
    
    # Gerar Kernel
    out_raster_name = f"Qualidade_Agua_{ano}"
    out_raster_path = os.path.join(gdb, out_raster_name)
    
    kd = KernelDensity(
        in_features=filtered_layer,
        population_field=None,
        cell_size=5,
        search_radius=200,
        area_unit_scale_factor="HECTARES",
        out_cell_values="DENSITIES",
        method="PLANAR"
    )
    
    kd.save(out_raster_path)
    
    print(f"✅ Kernel {ano} gerado com sucesso.")

    # Salvar na Lista para adicionar no Contents
    rasters_gerados.append(out_raster_path)
    
print("✅ Todos os anos processados com sucesso.")

# Adicionando Rasters no Contents
for raster in rasters_gerados:
    mapa.addDataFromPath(raster)

# Excluindo Raster Redundante do Contents
for lyr in mapa.listLayers():
    if lyr.name.startswith("OS"):
        mapa.removeLayer(lyr)
        break


🔄 Processando ano 2015...
✅ Kernel 2015 gerado com sucesso.

🔄 Processando ano 2016...
✅ Kernel 2016 gerado com sucesso.

🔄 Processando ano 2017...
✅ Kernel 2017 gerado com sucesso.

🔄 Processando ano 2018...
✅ Kernel 2018 gerado com sucesso.

🔄 Processando ano 2019...
✅ Kernel 2019 gerado com sucesso.

🔄 Processando ano 2020...
✅ Kernel 2020 gerado com sucesso.

🔄 Processando ano 2021...
✅ Kernel 2021 gerado com sucesso.

🔄 Processando ano 2022...
✅ Kernel 2022 gerado com sucesso.

🔄 Processando ano 2023...
✅ Kernel 2023 gerado com sucesso.

🔄 Processando ano 2024...
✅ Kernel 2024 gerado com sucesso.

🔄 Processando ano 2025...
✅ Kernel 2025 gerado com sucesso.
✅ Todos os anos processados com sucesso.


In [7]:
# Filtro para 10 anos - OS Descarga

In [14]:
# Nome da camada no BD
input_layer = "BDGIS.Ordens_Servico"

# Expressão SQL de filtro
sql_filter = """
DATAEXECUCAOINICIO between '2015-01-01 00:00:00' and '2025-12-31 23:59:59'
AND 
DESCRICAORA in ('Aguas Claras', 'Agua Quente', 'Arniqueira', 'Brazlandia', 'Ceilandia', 'Ceilandia II',
'Gama', 'Recanto das Emas', 'Riacho Fundo', 'Riacho Fundo II', 'Samambaia', 'Santa Maria', 'Taguatinga')
AND 
MOTIVONAOEXECUCAO = 0
AND 
DESCSITUACAOOS = 'Baixada'
AND
SERVAPROPRIADO in ('8400000031803')
"""

# Criar camada temporária
filtered_layer = "OS_Filtradas_Descarga_2015_2025"
arcpy.MakeFeatureLayer_management(input_layer, filtered_layer, sql_filter)

<Result 'OS_Filtradas_Descarga_2015_2025'>

In [11]:
# Trabalhando dado de OS de Descarga - Intersect com AOI (RAs)

In [ ]:
# Dataset
feature_dataset = "Qualidade_da_Agua"

# Camadas de entrada
input_features = ["OS_Filtradas_Descarga_2015_2025", "AOI_Sul_Oeste_v4"]

# Nome da feature class de saída
output_name = "OS_Descarga_2015_2025_AOI"

# Caminho completo da saída (dataset)
output_feature = os.path.join(gdb, feature_dataset, output_name)

# Executar Intersect
arcpy.analysis.Intersect(
    in_features=input_features,
    out_feature_class=output_feature,
    join_attributes="ALL",
    cluster_tolerance=None,
    output_type="INPUT"
)

print("✅ Intersect realizado com sucesso.")

# Campos que queremos manter
campos_manter = ["RA_Python", "SERVAPROPRIADO", "DESCRICAORA", "DATAEXECUCAOINICIO"]

# Listar todos os campos da camada
fields = arcpy.ListFields(output_feature)

# Criar lista de campos para excluir
campos_excluir = [
    f.name for f in fields
    if f.name not in campos_manter and not f.required
]

# Excluir campos
if campos_excluir:
    arcpy.management.DeleteField(output_feature, campos_excluir)

print("✅ Campos desnecessários removidos.")

In [ ]:
# Criando a Coluna "Ano"

In [17]:
# Entrada
input_feature = "OS_Descarga_2015_2025_AOI"

# Nome do novo campo
campo_ano = "Ano"

# Criar campo (tipo SHORT ou LONG)
arcpy.management.AddField(input_feature, campo_ano, "SHORT")

# Calcular o ano a partir da data
arcpy.management.CalculateField(
    in_table=input_feature,
    field=campo_ano,
    expression="!DATAEXECUCAOINICIO!.year",
    expression_type="PYTHON3"
)

print("Campo ANO criado e calculado com sucesso.")

Campo ANO criado e calculado com sucesso.


In [3]:
# Cortando Rasters por RA

In [5]:
aoi_fc = "AOI_Sul_Oeste_v4"
name_field = "RA_Python"

# Novo gdb para salvar os dados por RA
gdb = r"F:\Projetos_ArcGIS_Gama\Qualidade_da_Agua\Rasters_por_RA.gdb"

# Lista para adicionar os rasters no Contents ao final do processo
rasters_gerados = []

anos = range(2015, 2026)

# Processamento
for ano in anos:
    
    print(f"\n🔄 Processando ano {ano}")
    
    # Raster já gerado anteriormente
    input_raster = f"Qualidade_Agua_{ano}"
    
    with arcpy.da.SearchCursor(aoi_fc, ["SHAPE@", name_field]) as cursor:
        for geom, nome in cursor:
            
            # Nome estruturado
            out_name = f"Qualidade_Agua_{ano}_{nome}"
            out_path = os.path.join(gdb, out_name)
            
            # Recorte do raster
            out_extract = ExtractByMask(input_raster, geom)
            out_extract.save(out_path)

            # Salvar na lista para adicionar no Contents depois
            rasters_gerados.append(out_path)

print("✅ Todos os anos processados com sucesso.")

# Adicionando Rasters Cortados no Contents
for raster in rasters_gerados:
    mapa.addDataFromPath(raster)

# Excluindo Rasters Originais do Contents
for lyr in mapa.listLayers():
    if (lyr.name.startswith("Qualidade_Agua_") and lyr.name.count("_") == 2):
        mapa.removeLayer(lyr)

# Excluindo Raster Redundante do Contents
for lyr in mapa.listLayers():
    if lyr.name.startswith("out_extract"):
        mapa.removeLayer(lyr)
        break


🔄 Processando ano 2015

🔄 Processando ano 2016

🔄 Processando ano 2017

🔄 Processando ano 2018

🔄 Processando ano 2019

🔄 Processando ano 2020

🔄 Processando ano 2021

🔄 Processando ano 2022

🔄 Processando ano 2023

🔄 Processando ano 2024

🔄 Processando ano 2025
✅ Todos os anos processados com sucesso.


In [7]:
# Atribuindo Simbologia

In [13]:
# Pasta onde estão salvos os arquivos .lyrx
lyrx_folder = r"F:\Projetos_ArcGIS_Gama\Qualidade_da_Agua\Lyrx_Qualidade_da_Agua"

# Mapeamento: RA -> arquivo base de simbologia (.lyrx)
mapping = {
    "Aguas_Claras": "Aguas_Claras_Qualidade_Agua_2023",
    "Agua_Quente": "Agua_Quente_Qualidade_Agua_2023",
    "Arniqueira": "Arniqueira_Qualidade_Agua_2023",
    "Brazlandia": "Brazlandia_Qualidade_Agua_2023",
    "Ceilandia": "Ceilandia_Qualidade_Agua_2023",
    "Engenho_das_Lajes": "Engenho_das_Lajes_Qualidade_Agua_2023",
    "Gama": "Gama_Qualidade_Agua_2023",
    "Incra_8": "Incra_8_Qualidade_Agua_2023",
    "Recanto_das_Emas": "Recanto_das_Emas_Qualidade_Agua_2023",
    "Riacho_Fundo": "Riacho_Fundo_Qualidade_Agua_2023",
    "Riacho_Fundo_II": "Riacho_Fundo_II_Qualidade_Agua_2023",
    "Samambaia": "Samambaia_Qualidade_Agua_2023",
    "Santa_Maria": "Santa_Maria_Qualidade_Agua_2023",
    "Sol_Nascente_e_Por_do_Sol": "Sol_Nascente_e_Por_do_Sol_Qualidade_Agua_2023",
    "Taguatinga": "Taguatinga_Qualidade_Agua_2023",
    "Area_Sul": "Area_Sul_Qualidade_Agua_2023",
    "Area_Oeste": "Area_Oeste_Qualidade_Agua_2023"
}

# Aplicando simbologia para todos os rasters
for lyr in mapa.listLayers():
    
    # Garantir que é Raster
    if not lyr.isRasterLayer:
        continue
    
    nome_layer = lyr.name
    
    # Extrair RA (última parte do nome)
    partes = nome_layer.split("_")
    
    ra_nome = "_".join(partes[3:])  # tudo depois do ano
    
    if ra_nome in mapping:
        
        sym_layer = mapping[ra_nome]
        symbology_file = os.path.join(lyrx_folder, sym_layer + ".lyrx")

        arcpy.management.ApplySymbologyFromLayer(lyr, symbology_file)

print("🎨 Processo de simbologia finalizado.")

🎨 Processo de simbologia finalizado.


In [18]:
# Exportando Layout - AOI

In [3]:
# Caminho do projeto
aprx_path = r"F:\Projetos_ArcGIS_Gama\Qualidade_da_Agua\Qualidade_da_Agua.aprx"
aprx = arcpy.mp.ArcGISProject(aprx_path)

# Layout e Pasta de Saída dos Mapas
layout_model = aprx.listLayouts("A1 - QUALIDADE DA ÁGUA")[0]
out_folder = r"F:\Projetos_ArcGIS_Gama\Qualidade_da_Agua\Mapas"

# Mapa e Map Frame do Layout
mapa_principal = aprx.listMaps("Mapa_Principal")[0]
map_frame = layout_model.listElements("MAPFRAME_ELEMENT", "Map_Frame_Principal")[0]

# Textos Dinâmicos
selo_elemento = layout_model.listElements("TEXT_ELEMENT", "Texto_Selo")[0]
ano_selo = layout_model.listElements("TEXT_ELEMENT", "Ano")[0]
ano_caps = layout_model.listElements("TEXT_ELEMENT", "Ano_Caps")[0]

# Bookmarks
bookmarks = mapa_principal.listBookmarks()

# Camadas que serão reativadas
camadas_urbanismo = ["Via_por_AOI_v4", "Quadras_Label_Mapas_de_Calor"]

# Loop
for bm in bookmarks:
    
    nome_ra = bm.name
    print(f"\n📍 Processando: {nome_ra}")
    
    # Encontrar todos os rasters da RA
    rasters_ra = [
        lyr for lyr in mapa_principal.listLayers()
        if lyr.isRasterLayer
        and lyr.name.startswith("Qualidade_Agua_")
        and nome_ra in lyr.name
    ]
    
    for raster_layer in rasters_ra:
        
        # Extrair ano do nome
        partes = raster_layer.name.split("_")
        ano = partes[2]
        
        # Desligar todas as camadas
        for lyr in mapa_principal.listLayers():
            lyr.visible = False
        
        # Ligar raster do ano atual
        raster_layer.visible = True
        
        # Camada de pontos de Descarga
        descarga_layer = [lyr for lyr in mapa_principal.listLayers() if lyr.name == "Descarga"]

        if descarga_layer:
            descarga_layer = descarga_layer[0]
            descarga_layer.visible = True
            descarga_layer.definitionQuery = f"Ano = {ano} AND RA_Python = '{nome_ra}'"
        
        # Urbanismo + filtro + selo
        selo_texto = nome_ra
        
        for urb_name in camadas_urbanismo:
            
            urb_layer = [lyr for lyr in mapa_principal.listLayers() if lyr.name == urb_name]
            
            if urb_layer:
                urb_layer = urb_layer[0]
                urb_layer.visible = True
                urb_layer.definitionQuery = f"RA_Python = '{nome_ra}'"
                
                # Buscar selo
                if urb_name == "Quadras_Label_Mapas_de_Calor":
                    with arcpy.da.SearchCursor(
                        urb_layer.dataSource,
                        ["RA_Python", "Selo"]
                    ) as cursor:
                        for row in cursor:
                            if row[0] == nome_ra:
                                selo_texto = row[1]
                                break
        
        # Atualizar textos no layout
        selo_elemento.text = selo_texto
        ano_selo.text = ano
        ano_caps.text = ano
        
        # Aplicar bookmark
        map_frame.zoomToBookmark(bm)
        
        # Nome do arquivo com RA + ano
        out_png = os.path.join(
            out_folder,
            f"{nome_ra}_{ano}.png"
        )
        
        layout_model.exportToPNG(out_png, resolution=300)

print("\n🏁 Processo de exportação finalizado.")


📍 Processando: Gama

📍 Processando: Aguas_Claras

📍 Processando: Taguatinga

📍 Processando: Brazlandia

📍 Processando: Samambaia

📍 Processando: Sol_Nascente_e_Por_do_Sol

📍 Processando: Ceilandia

📍 Processando: Santa_Maria

📍 Processando: Recanto_das_Emas

📍 Processando: Arniqueira

📍 Processando: Incra_8

📍 Processando: Engenho_das_Lajes

📍 Processando: Agua_Quente

📍 Processando: Riacho_Fundo_II

📍 Processando: Riacho_Fundo

🏁 Processo de exportação finalizado.


In [4]:
# Exportando Layout - Área Sul e Oeste 

In [5]:
# Caminho do projeto
aprx_path = r"F:\Projetos_ArcGIS_Gama\Qualidade_da_Agua\Qualidade_da_Agua.aprx"
aprx = arcpy.mp.ArcGISProject(aprx_path)

# Layout e Pasta de Saída dos Mapas
layout_model = aprx.listLayouts("A1 - QUALIDADE DA ÁGUA")[0]
out_folder = r"F:\Projetos_ArcGIS_Gama\Qualidade_da_Agua\Mapas"

# Mapa e Map Frame do Layout
mapa_principal = aprx.listMaps("Mapa_Principal")[0]
map_frame = layout_model.listElements("MAPFRAME_ELEMENT", "Map_Frame_Principal")[0]

# Textos Dinâmicos
selo_elemento = layout_model.listElements("TEXT_ELEMENT", "Texto_Selo")[0]
ano_selo = layout_model.listElements("TEXT_ELEMENT", "Ano")[0]
ano_caps = layout_model.listElements("TEXT_ELEMENT", "Ano_Caps")[0]

# Bookmarks
bookmarks = mapa_principal.listBookmarks()

# Camadas que serão reativadas
camadas_urbanismo = ["Via_por_AOI_v4_Sul_Oeste", "Quadras_Label_Mapas_de_Calor_Sul_Oeste"]

# Loop
for bm in bookmarks:
    
    nome_ra = bm.name
    print(f"\n📍 Processando: {nome_ra}")
    
    # Encontrar todos os rasters da RA
    rasters_ra = [
        lyr for lyr in mapa_principal.listLayers()
        if lyr.isRasterLayer
        and lyr.name.startswith("Qualidade_Agua_")
        and nome_ra in lyr.name
    ]
    
    for raster_layer in rasters_ra:
        
        # Extrair ano do nome
        partes = raster_layer.name.split("_")
        ano = partes[2]
        
        # Desligar todas as camadas
        for lyr in mapa_principal.listLayers():
            lyr.visible = False
        
        # Ligar raster do ano atual
        raster_layer.visible = True
        
        # Camada de pontos de Descarga
        descarga_layer = [lyr for lyr in mapa_principal.listLayers() if lyr.name == "Descarga_Sul_Oeste"]

        if descarga_layer:
            descarga_layer = descarga_layer[0]
            descarga_layer.visible = True
            descarga_layer.definitionQuery = f"Ano = {ano} AND RA_Python = '{nome_ra}'"
        
        # Urbanismo + filtro + selo
        selo_texto = nome_ra
        
        for urb_name in camadas_urbanismo:
            
            urb_layer = [lyr for lyr in mapa_principal.listLayers() if lyr.name == urb_name]
            
            if urb_layer:
                urb_layer = urb_layer[0]
                urb_layer.visible = True
                urb_layer.definitionQuery = f"RA_Python = '{nome_ra}'"
                
                # Buscar selo
                if urb_name == "Quadras_Label_Mapas_de_Calor_Sul_Oeste":
                    with arcpy.da.SearchCursor(
                        urb_layer.dataSource,
                        ["RA_Python", "Selo"]
                    ) as cursor:
                        for row in cursor:
                            if row[0] == nome_ra:
                                selo_texto = row[1]
                                break
        
        # Atualizar textos no layout
        selo_elemento.text = selo_texto
        ano_selo.text = ano
        ano_caps.text = ano
        
        # Aplicar bookmark
        map_frame.zoomToBookmark(bm)
        
        # Nome do arquivo com RA + ano
        out_png = os.path.join(
            out_folder,
            f"{nome_ra}_{ano}.png"
        )
        
        layout_model.exportToPNG(out_png, resolution=300)

print("\n🏁 Processo de exportação finalizado.")


📍 Processando: Area_Sul

📍 Processando: Area_Oeste

🏁 Processo de exportação finalizado.
